In [2]:
import os
import pandas as pd
import numpy as np
import logging
from pathlib import Path
import scipy
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline


In [3]:
# == Configuration ==
# === Logging format: timestamp - level - message ===
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# === Paths ===
CURRENT_DIR = Path.cwd()
PATH_RESULTS = CURRENT_DIR / 'results'
PATH_Z = CURRENT_DIR / 'temp_data' / 'latents_full.npz'
PATH_BEH = CURRENT_DIR / 'data' / 'processed' /'retained_trials_for_latents.csv'
PATH_CPP = CURRENT_DIR / 'data' / 'processed' /'resp_locked_eeg_retained_trials.npy'


In [4]:
# load data
data_z = np.load(PATH_Z, allow_pickle=True)['Z']
data_eeg = np.load(PATH_CPP, allow_pickle=True)
data_cpp = np.nanmean(data_eeg, axis=1)
data_behavior = pd.read_csv(PATH_BEH)
assert len(data_z) == len(data_behavior) == len(data_cpp), f" feature and behavior data are unaligned: fearture:{len(data_z)}; behavior data:{len(data_behavior)}; cpp data:{len(data_cpp)}"
#data_behavior["subj_num"] = pd.factorize(data_behavior["subj_id"])[0] + 1
subj_dummies = pd.get_dummies(data_behavior["subject_id"], drop_first=True).values.astype(np.float32)

vals = data_behavior['cue_dimensionality'].values.astype(np.float32)
std = vals.std()
cue_dimensionality = ((vals - vals.mean()) / std).reshape(-1, 1)

acc = np.array(data_behavior['probe_accuracy']).reshape(-1, 1)
data_behavior['log_rt'] = np.log(data_behavior['probe_rt'])

# M0 baseline：use condition to predict rt

In [6]:
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# correlate each latent dimension with probe_rt
rt = pd.to_numeric(data_behavior['log_rt'], errors='coerce').to_numpy()
data_m0 = np.hstack([cue_dimensionality, subj_dummies, acc]).astype(np.float32)

# 做data_z_avg和rt的ridge regression
X_train, X_test, y_train, y_test = train_test_split(data_m0, rt, test_size=0.2, random_state=42)

alphas = np.logspace(-3, 3, 50)

model = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas, cv=5))  # 5 折交叉验证
model.fit(X_train, y_train)
ridge_cv = model.named_steps["ridgecv"]
print("最优 alpha:", ridge_cv.alpha_)
print("测试集 R²:", model.score(X_test, y_test))


最优 alpha: 79.06043210907701
测试集 R²: 0.20468645100397975


# M1: add CPP slopes

In [14]:
# Calculate CPP slopes
time_windows = 0.82, 0.92
time_windows = 0.4, 0.7
sample_rate = 256
data4slope = data_cpp[:, int(time_windows[0] * sample_rate):int(time_windows[1] * sample_rate)]
n_trial = data4slope.shape[0]
time_vector = np.arange(data4slope.shape[1]) / sample_rate
slope = np.array([np.polyfit(time_vector, data4slope[i, :], deg=1)[0] for i in range(n_trial)]) 
var_slope = pd.DataFrame(
    {"cpp_slope": slope}
    )

# M1: add CPP slopes
rt = pd.to_numeric(data_behavior['log_rt'], errors='coerce').to_numpy()
var_beh = np.hstack([cue_dimensionality, subj_dummies, acc]).astype(np.float32)
assert len(var_slope) == len(var_beh), f"var_slope and var_beh have different lengths: {len(var_slope)} and {len(var_beh)}"
data_m1 = np.hstack([var_beh, var_slope])

# 做data_z_avg和rt的ridge regression
X_train, X_test, y_train, y_test = train_test_split(data_m1, rt, test_size=0.2, random_state=42)

alphas = np.logspace(-3, 3, 50)

model = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas, cv=5))  # 5 折交叉验证
model.fit(X_train, y_train)
ridge_cv = model.named_steps["ridgecv"]
print("最优 alpha:", ridge_cv.alpha_)
print("测试集 R²:", model.score(X_test, y_test))


最优 alpha: 79.06043210907701
测试集 R²: 0.22428671493425445


# M2: add CPP ams 

In [16]:
# Calculate CPP amplitude
time_windows = 0.82, 0.92
time_windows = 0.4, 0.7
sample_rate = 256
data4ams = data_cpp[:, int(time_windows[0] * sample_rate):int(time_windows[1] * sample_rate)]
ams = np.array([np.nanmean(data4ams[i, :]) for i in range(n_trial)]) 
var_ams = pd.DataFrame(
    {"cpp_ams": ams}
    )

# M2: add CPP amplitude
rt = pd.to_numeric(data_behavior['log_rt'], errors='coerce').to_numpy()
var_beh = np.hstack([cue_dimensionality, subj_dummies, acc]).astype(np.float32)
assert len(var_ams) == len(var_beh), f"var_ams and var_beh have different lengths: {len(var_ams)} and {len(var_beh)}"
data_m2 = np.hstack([var_beh, var_ams])

# 做cpp_ams和rt的ridge regression
X_train, X_test, y_train, y_test = train_test_split(data_m2, rt, test_size=0.2, random_state=42)

alphas = np.logspace(-3, 3, 50)

model = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas, cv=5))  # 5 折交叉验证
model.fit(X_train, y_train)
ridge_cv = model.named_steps["ridgecv"]
print("最优 alpha:", ridge_cv.alpha_)
print("测试集 R²:", model.score(X_test, y_test))


最优 alpha: 59.636233165946365
测试集 R²: 0.22607330924087232


# M3: add CPP peak amplitude

In [17]:
# Calculate CPP peak amplitude
time_windows = 0.95, 1.05
time_windows = 0.4, 0.7
sample_rate = 256
data4pams = data_cpp[:, int(time_windows[0] * sample_rate):int(time_windows[1] * sample_rate)]
pams = np.array([np.nanmax(data4pams[i, :]) for i in range(n_trial)]) 
var_pams = pd.DataFrame(
    {"cpp_pams": pams}
    )

# M3: add CPP peak amplitude
rt = pd.to_numeric(data_behavior['log_rt'], errors='coerce').to_numpy()
var_beh = np.hstack([cue_dimensionality, subj_dummies, acc]).astype(np.float32)
assert len(var_pams) == len(var_beh), f"var_pams and var_beh have different lengths: {len(var_pams)} and {len(var_beh)}"
data_m3 = np.hstack([var_beh, var_pams])

# 做cpp_pams和rt的ridge regression  
X_train, X_test, y_train, y_test = train_test_split(data_m1, rt, test_size=0.2, random_state=42)

alphas = np.logspace(-3, 3, 50)

model = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas, cv=5))  # 5 折交叉验证
model.fit(X_train, y_train)
ridge_cv = model.named_steps["ridgecv"]
print("最优 alpha:", ridge_cv.alpha_)
print("测试集 R²:", model.score(X_test, y_test))


最优 alpha: 79.06043210907701
测试集 R²: 0.22428671493425445


# M4: add 3 CPP features (cpp_ams, cpp_pams, cpp_slope)

In [18]:
# Calculate CPP peak amplitude
time_windows_slps_ams = 0.95, 1.05
time_windows_pams = 0.82, 0.92
time_windows = 0.4, 0.7
sample_rate = 256
data4pams = data_cpp[:, int(time_windows_pams[0] * sample_rate):int(time_windows_pams[1] * sample_rate)]
data4ams = data_cpp[:, int(time_windows_slps_ams[0] * sample_rate):int(time_windows_slps_ams[1] * sample_rate)]
data4slope = data_cpp[:, int(time_windows_slps_ams[0] * sample_rate):int(time_windows_slps_ams[1] * sample_rate)]
pams = np.array([np.nanmax(data4pams[i, :]) for i in range(n_trial)]) 
ams = np.array([np.nanmean(data4ams[i, :]) for i in range(n_trial)])
n_trial = data4slope.shape[0]
time_vector = np.arange(data4slope.shape[1]) / sample_rate
slope = np.array([np.polyfit(time_vector, data4slope[i, :], deg=1)[0] for i in range(n_trial)]) 
var_features = pd.DataFrame(
    {"cpp_pams": pams, "cpp_ams": ams, "cpp_slope": slope}
    )

# M4: add CPP peak amplitude
rt = pd.to_numeric(data_behavior['log_rt'], errors='coerce').to_numpy()
var_beh = np.hstack([cue_dimensionality, subj_dummies, acc]).astype(np.float32)
assert len(var_features) == len(var_beh), f"var_features and var_beh have different lengths: {len(var_features)} and {len(var_beh)}"
data_m4 = np.hstack([var_beh, var_features])

# 做cpp_pams, cpp_ams, cpp_slope和rt的ridge regression  
X_train, X_test, y_train, y_test = train_test_split(data_m4, rt, test_size=0.2, random_state=42)

alphas = np.logspace(-3, 3, 50)

model = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas, cv=5))  # 5 折交叉验证
model.fit(X_train, y_train)
ridge_cv = model.named_steps["ridgecv"]
print("最优 alpha:", ridge_cv.alpha_)
print("测试集 R²:", model.score(X_test, y_test))


最优 alpha: 104.81131341546852
测试集 R²: 0.2324986696592688


# M5 add latent variable from GRU model

In [21]:
# average targeted timestampes
## epoch targeted timeswindow
SAMPLE_RATE = 256
TIME_WINDOW = (0.88, 0.92)# timewondow was selected according to  O'Connell et al (2012), in which the timewindow was used for calculate the slope of CPP
TIME_WINDOW = (0.2, 0.3)
time_windows = 0.4, 0.7
# Convert seconds to sample indices
start_idx = int(round(TIME_WINDOW[0] * SAMPLE_RATE))
end_idx = int(round(TIME_WINDOW[1] * SAMPLE_RATE))

# Extract z values in the target time window
data_z_target = data_z[:, start_idx:end_idx, :]

# Average z within the target time window
data_z_avg = np.nanmean(data_z_target, axis=1)
data_m5 = np.hstack([data_z_avg, var_features,var_beh])
data_m5 = np.hstack([data_z_avg,var_beh])
data_m5 = np.hstack([data_z_avg])
# correlate each latent dimension with probe_rt
rt = pd.to_numeric(data_behavior['log_rt'], errors='coerce').to_numpy()

# 做data_z_avg和rt的ridge regression
X_train, X_test, y_train, y_test = train_test_split(data_m5, rt, test_size=0.2, random_state=42)

alphas = np.logspace(-3, 3, 50)

model = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas, cv=5))  # 5 折交叉验证
model.fit(X_train, y_train)
ridge_cv = model.named_steps["ridgecv"]
print("最优 alpha:", ridge_cv.alpha_)
print("测试集 R²:", model.score(X_test, y_test))


最优 alpha: 33.9322177189533
测试集 R²: 0.15526314872143432


# 以下cell为备份不需要看

# 以下cell为备份不需要看

# 以下cell为备份不需要看

In [62]:
# average targeted timestampes
## epoch targeted timeswindow
SAMPLE_RATE = 256
TIME_WINDOW = (0.88, 0.92)# timewondow was selected according to  O'Connell et al (2012), in which the timewindow was used for calculate the slope of CPP
# Convert seconds to sample indices
start_idx = int(round(TIME_WINDOW[0] * SAMPLE_RATE))
end_idx = int(round(TIME_WINDOW[1] * SAMPLE_RATE))

# Extract z values in the target time window
data_z_target = data_z[:, start_idx:end_idx, :]

# Average z within the target time window
data_z_avg = np.nanmean(data_z_target, axis=1)


理论上说，CPP得slopes，amplitude或者peak amplitude是可以预测rt，acc的。slope越大，rt越小。GRU的预测能力是不是比它好？比它好的话可能有几种情况：1）GRU的特征维度中包含了多种信息，这个多种信息比单个信息的预测能力更好；2）如果比三个我们目前认为的比较好的信息的结合的预测能力还要好的话，说明CPP的数据中还有没有被我们考虑到的信息，这个信息可能也表现了证据累积。
除了rt以外，还有其他变量也会被使用。

一个逻辑是，现在大家说CPP符合，就是看CPP是不是在不同rt，acc上有差异。如果我们发现CPP得到的特征值对rt的预测能力更好，我们可以说简单地做rt的比较就说CPP符合预期是不充分，可能是由其他因素导致的这个现象。

### CPP中可能还有其他变量，这些变量可能也表现了证据累积 ###

其他研究中，发现CPP在不同rt中有差异等等，但是也意识到它们混淆了其他因素
在找是否有研究简单就用rt有差异就简单验证在它们的研究中CPP表征了证据累积

下一步：

# 暂时不做z-score

ridge regression的逻辑是：

先用cross-validation找到最优的alpha；

然后用最优的alpha训练模型。然后用这个模型预测测试集。得到R²。
 - R²的解释是：模型解释了数据多少%的方差。
 - R²的取值范围是0到1，1表示模型解释了数据的所有方差，0表示模型解释了0%的方差。

最后得到模型的系数，这个系数是模型对每个特征的权重。

model = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas, cv=5))  # 5 折交叉验证

model.fit(X_train, y_train)

用这个代码构建四个模型并进行比较

In [63]:
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# correlate each latent dimension with probe_rt
rt = pd.to_numeric(data_behavior['probe_rt'], errors='coerce').to_numpy()

# 做data_z_avg和rt的ridge regression
X_train, X_test, y_train, y_test = train_test_split(data_z_avg, rt, test_size=0.2, random_state=42)

alphas = np.logspace(-3, 3, 50)

model = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas, cv=5))  # 5 折交叉验证
model.fit(X_train, y_train)
ridge_cv = model.named_steps["ridgecv"]
print("最优 alpha:", ridge_cv.alpha_)
print("测试集 R²:", model.score(X_test, y_test))


最优 alpha: 33.9322177189533
测试集 R²: 0.030660396196027118


In [64]:
ridge_cv.coef_

array([ 0.05587967, -0.01271862, -0.02175787,  0.05431388,  0.00011951,
        0.00270688, -0.00661299,  0.0561066 ,  0.01576189,  0.00591649,
       -0.01999559, -0.05316192,  0.02695216,  0.04995903, -0.03785292,
       -0.00424344, -0.00112528,  0.02555316,  0.02897073,  0.00032461,
        0.02845725, -0.0253772 ,  0.00326905,  0.0065716 ,  0.01078838,
        0.00694333,  0.0508675 , -0.03651111,  0.01035569, -0.01414377,
       -0.02725149, -0.00673002], dtype=float32)

In [65]:
model   

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('ridgecv', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,32
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value
"mean_ mean_: ndarray of shape (n_features,) or NoneThe mean value for each feature in the training set.Equal to ``None`` when ``with_mean=False`` and ``with_std=False``.","ndarray[float64](32,)","[-0.13,-0.75, 0.64,..., 0.91,-0.85,-0.93]"


In [66]:
# correlate each latent dimension with probe_rt
probe_rt = pd.to_numeric(data_behavior['probe_rt'], errors='coerce').to_numpy()

corr_results = []

for dim_idx in range(data_z_avg.shape[1]):
    z_dim = data_z_avg[:, dim_idx]
    valid_mask = np.isfinite(z_dim) & np.isfinite(probe_rt)

    if valid_mask.sum() < 3:
        corr_value = np.nan
    else:
        corr_value = np.corrcoef(z_dim[valid_mask], probe_rt[valid_mask])[0, 1]

    corr_results.append({
        'dimension': dim_idx,
        'pearson_r_with_probe_rt': corr_value,
        'abs_pearson_r': np.abs(corr_value) if np.isfinite(corr_value) else np.nan,
        'n_valid_trials': int(valid_mask.sum())
    })

corr_results = pd.DataFrame(corr_results)
corr_results = corr_results.sort_values('abs_pearson_r', ascending=False).reset_index(drop=True)

display(corr_results)

best_row = corr_results.iloc[0]
print('dimension with highest |correlation| with probe_rt:')
print(f"dimension = {int(best_row['dimension'])}")
print(f"pearson r = {best_row['pearson_r_with_probe_rt']:.6f}")
print(f"number of valid trials = {int(best_row['n_valid_trials'])}")


,dimension,pearson_r_with_probe_rt,abs_pearson_r,n_valid_trials
0,14,-0.155141,0.155141,7297
1,21,0.148493,0.148493,7297
2,6,0.147631,0.147631,7297
3,0,0.142797,0.142797,7297
4,9,0.135631,0.135631,7297
5,10,0.127066,0.127066,7297
6,3,0.126906,0.126906,7297
7,4,-0.122249,0.122249,7297
8,19,-0.109695,0.109695,7297
9,29,0.107850,0.107850,7297


dimension with highest |correlation| with probe_rt:
dimension = 14
pearson r = -0.155141
number of valid trials = 7297


In [67]:
# test whether the correlation of each latent dimension with probe_rt is significant
from scipy.stats import pearsonr

sig_results = []

for dim_idx in range(data_z_avg.shape[1]):
    z_dim = data_z_avg[:, dim_idx]
    valid_mask = np.isfinite(z_dim) & np.isfinite(probe_rt)

    if valid_mask.sum() < 3:
        r_value = np.nan
        p_value = np.nan
    else:
        r_value, p_value = pearsonr(z_dim[valid_mask], probe_rt[valid_mask])

    sig_results.append({
        'dimension': dim_idx,
        'pearson_r_with_probe_rt': r_value,
        'p_value': p_value,
        'significant_p_lt_0_05': bool(p_value < 0.05) if np.isfinite(p_value) else False,
        'significant_p_lt_0_01': bool(p_value < 0.01) if np.isfinite(p_value) else False,
        'n_valid_trials': int(valid_mask.sum())
    })

sig_results = pd.DataFrame(sig_results)
sig_results['abs_pearson_r'] = sig_results['pearson_r_with_probe_rt'].abs()
sig_results = sig_results.sort_values('abs_pearson_r', ascending=False).reset_index(drop=True)

display(sig_results)

best_sig_row = sig_results.iloc[0]
print('dimension with highest |correlation| and its significance:')
print(f"dimension = {int(best_sig_row['dimension'])}")
print(f"pearson r = {best_sig_row['pearson_r_with_probe_rt']:.6f}")
print(f"p value = {best_sig_row['p_value']:.6e}")
print(f"p < 0.05: {bool(best_sig_row['significant_p_lt_0_05'])}")
print(f"p < 0.01: {bool(best_sig_row['significant_p_lt_0_01'])}")


,dimension,pearson_r_with_probe_rt,p_value,significant_p_lt_0_05,significant_p_lt_0_01,n_valid_trials,abs_pearson_r
0,14,-0.155141,1.527258e-40,True,True,7297,0.155141
1,21,0.148493,2.991111e-37,True,True,7297,0.148493
2,6,0.147631,7.789750e-37,True,True,7297,0.147631
3,0,0.142797,1.506625e-34,True,True,7297,0.142797
4,9,0.135631,2.648716e-31,True,True,7297,0.135631
5,10,0.127066,1.196144e-27,True,True,7297,0.127066
6,3,0.126906,1.392424e-27,True,True,7297,0.126906
7,4,-0.122249,1.063748e-25,True,True,7297,0.122249
8,19,-0.109695,5.606619e-21,True,True,7297,0.109695
9,29,0.107850,2.506212e-20,True,True,7297,0.107850


dimension with highest |correlation| and its significance:
dimension = 14
pearson r = -0.155141
p value = 1.527258e-40
p < 0.05: True
p < 0.01: True


In [ ]:
# Time generalization: scan hidden-state windows and test incremental value for log RT
import matplotlib.pyplot as plt

latent_bundle = np.load(PATH_Z, allow_pickle=True)
latent_times_ms = latent_bundle['times_ms'].astype(float)

window_size_ms = 100
step_ms = 25
window_starts = np.arange(latent_times_ms.min(), latent_times_ms.max() - window_size_ms + step_ms, step_ms)

row_index = np.arange(len(rt))
train_idx, test_idx = train_test_split(row_index, test_size=0.2, random_state=42)

baseline_model = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas, cv=5))
baseline_model.fit(var_beh[train_idx], rt[train_idx])
baseline_r2 = baseline_model.score(var_beh[test_idx], rt[test_idx])

time_generalization_rows = []

for start_ms in window_starts:
    end_ms = start_ms + window_size_ms
    mask = (latent_times_ms >= start_ms) & (latent_times_ms < end_ms)

    if mask.sum() < 2:
        continue

    z_window = np.nanmean(data_z[:, mask, :], axis=1)
    design = np.hstack([z_window, var_beh]).astype(np.float32)

    model_window = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas, cv=5))
    model_window.fit(design[train_idx], rt[train_idx])
    model_r2 = model_window.score(design[test_idx], rt[test_idx])

    time_generalization_rows.append({
        'start_ms': float(start_ms),
        'end_ms': float(end_ms),
        'center_ms': float((start_ms + end_ms) / 2.0),
        'n_timepoints': int(mask.sum()),
        'baseline_r2': float(baseline_r2),
        'baseline_plus_hidden_r2': float(model_r2),
        'delta_r2': float(model_r2 - baseline_r2),
        'best_alpha': float(model_window.named_steps['ridgecv'].alpha_),
    })

time_generalization_results = pd.DataFrame(time_generalization_rows)
time_generalization_results = time_generalization_results.sort_values('center_ms').reset_index(drop=True)

best_time_window = time_generalization_results.loc[time_generalization_results['delta_r2'].idxmax()]
display(time_generalization_results.head())
display(time_generalization_results.sort_values('delta_r2', ascending=False).head(10))

print('baseline test R^2 =', baseline_r2)
print(
    f"best hidden window: {best_time_window['start_ms']:.1f} to {best_time_window['end_ms']:.1f} ms "
    f"(center = {best_time_window['center_ms']:.1f} ms), delta R^2 = {best_time_window['delta_r2']:.4f}"
)

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

axes[0].plot(
    time_generalization_results['center_ms'],
    time_generalization_results['baseline_plus_hidden_r2'],
    color='tab:blue',
    linewidth=2,
    label='baseline + hidden',
)
axes[0].axhline(baseline_r2, color='black', linestyle='--', linewidth=1.5, label='baseline only')
axes[0].axvline(0, color='red', linestyle=':', linewidth=1.2)
axes[0].scatter(best_time_window['center_ms'], best_time_window['baseline_plus_hidden_r2'], color='tab:orange', s=45, zorder=3)
axes[0].set_ylabel('Test R^2')
axes[0].set_title('Time generalization of hidden-state contribution to log RT')
axes[0].legend(frameon=False)

axes[1].plot(
    time_generalization_results['center_ms'],
    time_generalization_results['delta_r2'],
    color='tab:green',
    linewidth=2,
)
axes[1].axhline(0, color='black', linestyle='--', linewidth=1.2)
axes[1].axvline(0, color='red', linestyle=':', linewidth=1.2)
axes[1].scatter(best_time_window['center_ms'], best_time_window['delta_r2'], color='tab:orange', s=45, zorder=3)
axes[1].set_xlabel('Window center (ms, response-locked)')
axes[1].set_ylabel('Delta R^2 vs baseline')

plt.tight_layout()
plt.show()
